#Imports

In [3]:
import sklearn
print(sklearn.__version__)


1.8.0


In [4]:
# Core Python
import os
import warnings

# Data handling
import numpy as np
import pandas as pd

# Visualization (EDA + sanity checks)
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
sns.set(style="whitegrid")

# Warnings
warnings.filterwarnings("ignore")

Confirming Target Variable

In [5]:
df = pd.read_csv("diabetic_data.csv")
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
df.shape
df.dtypes
df.isna().mean().sort_values(ascending=False).head(10)

max_glu_serum    0.947468
A1Cresult        0.832773
encounter_id     0.000000
nateglinide      0.000000
glimepiride      0.000000
acetohexamide    0.000000
glipizide        0.000000
glyburide        0.000000
tolbutamide      0.000000
pioglitazone     0.000000
dtype: float64

In [7]:
df["readmit_30d"] = df["readmitted"].map({
    "<30": 1,
    ">30": 0,
    "NO": 0
})

df["readmit_30d"].value_counts(normalize=True)

readmit_30d
0    0.888401
1    0.111599
Name: proportion, dtype: float64

In [8]:
df = df.drop(columns=["readmitted"])

In [9]:
# Class imbalance check
positive_rate = df["readmit_30d"].mean()
print(f"30-day readmission rate: {positive_rate:.2%}")

30-day readmission rate: 11.16%


The outcome is moderately imbalanced, motivating the use of AUC and recall-focused evaluation.